# Model Training

### Importing data and required packages

In [16]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, recall_score, precision_score

### Import csv data as pandas dataframe

In [17]:
df = pd.read_csv("../data/Fertilizer Prediction.csv")

### Show top 5 records

In [20]:
df.head()

,Temperature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,26,52,38,Sandy,Maize,37,0,0,Urea
1,29,52,45,Loamy,Sugarcane,12,0,36,DAP
2,34,65,62,Black,Cotton,7,9,30,14-35-14
3,32,62,34,Red,Tobacco,22,0,20,28-28
4,28,54,46,Clayey,Paddy,35,0,0,Urea


In [19]:
df.rename(columns = {'Temparature': 'Temperature', 'Humidity ': 'Humidity'}, inplace = True)

### Preparing X and Y variables

In [21]:
X = df.drop('Fertilizer Name', axis = 1)

In [22]:
X

,Temperature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous
0,26,52,38,Sandy,Maize,37,0,0
1,29,52,45,Loamy,Sugarcane,12,0,36
2,34,65,62,Black,Cotton,7,9,30
3,32,62,34,Red,Tobacco,22,0,20
4,28,54,46,Clayey,Paddy,35,0,0
...,...,...,...,...,...,...,...,...
94,25,50,32,Clayey,Pulses,24,0,19
95,30,60,27,Red,Tobacco,4,17,17
96,38,72,51,Loamy,Wheat,39,0,0
97,36,60,43,Sandy,Millets,15,0,41


In [23]:
y = df['Fertilizer Name']

In [24]:
y

0         Urea
1          DAP
2     14-35-14
3        28-28
4         Urea
        ...   
94       28-28
95    10-26-26
96        Urea
97         DAP
98       20-20
Name: Fertilizer Name, Length: 99, dtype: str

In [25]:
print("Categories in 'Soil Type':", df['Soil Type'].unique())
print("Categories in 'Crop Type':", df['Crop Type'].unique())

Categories in 'Soil Type': <StringArray>
['Sandy', 'Loamy', 'Black', 'Red', 'Clayey']
Length: 5, dtype: str
Categories in 'Crop Type': <StringArray>
[      'Maize',   'Sugarcane',      'Cotton',     'Tobacco',       'Paddy',
      'Barley',       'Wheat',     'Millets',   'Oil seeds',      'Pulses',
 'Ground Nuts']
Length: 11, dtype: str


In [26]:
#column transformer
num_features = X.select_dtypes(exclude = "str").columns
cat_features = X.select_dtypes(include = "str").columns

numeric_transformer = StandardScaler()
ohe_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", ohe_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)

In [27]:
X = preprocessor.fit_transform(X)

In [28]:
X

array([[ 0.        ,  0.        ,  0.        , ...,  1.56753938,
        -0.58491041, -1.38760694],
       [ 0.        ,  0.        ,  1.        , ..., -0.59865825,
        -0.58491041,  1.29720909],
       [ 1.        ,  0.        ,  0.        , ..., -1.03189778,
         0.97077668,  0.84973976],
       ...,
       [ 0.        ,  0.        ,  1.        , ...,  1.74083519,
        -0.58491041, -1.38760694],
       [ 0.        ,  0.        ,  0.        , ..., -0.33871454,
        -0.58491041,  1.67010021],
       [ 1.        ,  0.        ,  0.        , ..., -0.59865825,
        -0.58491041, -0.64182471]], shape=(99, 22))

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y
)

In [30]:
X_train.shape, X_test.shape

((79, 22), (20, 22))

#### Create an Evaluate Function to give all metrics after model Training

In [31]:
def evaluate_model(true, predicted):
    accuracy = accuracy_score(true, predicted)
    cm = confusion_matrix(true, predicted)
    report = classification_report(true, predicted)
    return accuracy, cm, report

In [32]:
models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "K Nearest Neighbors": KNeighborsClassifier()
}
model_list = []
accuracy_list = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) #train the model
    
    #make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    #evaluate train and test dataset
    model_train_accuracy, model_train_cm, model_train_report = evaluate_model(y_train, y_train_pred)
    model_test_accuracy, model_test_cm, model_test_report = evaluate_model(y_test, y_test_pred)
    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print(f"- Accuracy: {model_train_accuracy:.4f}")
    print(f"- Confusion Matrix: \n {model_train_cm}")
    print(f"- Classification Report: \n {model_train_report}")
    
    print('----------------------------------')
    
    print('Model performance for Test set')
    print(f"- Accuracy: {model_test_accuracy:.4f}")
    print(f"- Confusion Matrix: \n {model_test_cm}")
    print(f"- Classification Report: \n {model_test_report}")
    accuracy_list.append(model_test_accuracy)
    
    print('='*35)
    print('\n')

Decision Tree
Model performance for Training set
- Accuracy: 1.0000
- Confusion Matrix: 
 [[ 6  0  0  0  0  0  0]
 [ 0 11  0  0  0  0  0]
 [ 0  0  6  0  0  0  0]
 [ 0  0  0 11  0  0  0]
 [ 0  0  0  0 14  0  0]
 [ 0  0  0  0  0 14  0]
 [ 0  0  0  0  0  0 17]]
- Classification Report: 
               precision    recall  f1-score   support

    10-26-26       1.00      1.00      1.00         6
    14-35-14       1.00      1.00      1.00        11
    17-17-17       1.00      1.00      1.00         6
       20-20       1.00      1.00      1.00        11
       28-28       1.00      1.00      1.00        14
         DAP       1.00      1.00      1.00        14
        Urea       1.00      1.00      1.00        17

    accuracy                           1.00        79
   macro avg       1.00      1.00      1.00        79
weighted avg       1.00      1.00      1.00        79

----------------------------------
Model performance for Test set
- Accuracy: 0.9500
- Confusion Matrix: 
 [[1 0 0 0 

As we can see from the output, Random forest is the best model